In [2]:
import sys
sys.path.insert(1, '../../.')
from params import p, p_cosmology
from transient_rates import phi_0, R_SFR, plot_rates

import numpy  as np
import matplotlib.pyplot as plt
import astropy.units as u
import scipy
import astropy.cosmology as acosmo
from firesong.Firesong import firesong_simulation

We want to reproduce the FireSong luminosity computation:

$$ \Phi_{z=1}^{PS} = \frac{4 \pi \Phi_\mathrm{diffuse}}
    {N_\mathrm{tot}\,d_L^2(z=1)\, \int_0^{10}
    \frac{ (1+z)^{-\gamma+2} }{d_L(z)^2}
    \frac{\frac{\mathrm{d}N}{\mathrm{d}z} V_c(z)}
    { \int_0^{z_\mathrm{max}} \frac{\mathrm{d}N}{\mathrm{d}z'}
    V_c(z') \,\mathrm{d}z'} \,\mathrm{d}z} $$

To make sure that that number is correct. 

In [3]:
def get_N_k_firesong(phi_z, R0, z_max=10, z_local=0.00023, p=p_cosmology, sky_area=4*np.pi*u.sr, time_window=None):

    R0 = R0.to(u.Mpc**-3 * u.yr ** -1)
    cosmo = acosmo.LambdaCDM(H0=p['h0']* u.km / u.s / u.Mpc, 
                             Om0=p['omega_M'],
                            Ode0=p['omega_L'])#, Tcmb0=2.725 * u.K,
    V_local = cosmo.comoving_volume(z_local)

    z_range_local = np.linspace(p['z_min'], z_local, 10000)
    z_range_max = np.linspace(p['z_min'], z_max, 10000)
    dVdz_max = cosmo.differential_comoving_volume(z_range_max)
    dVdz_local = cosmo.differential_comoving_volume(z_range_local)

    y_z_max = phi_z(z_range_max) * dVdz_max / (1 + z_range_max)
    y_z_local = phi_z(z_range_local) * dVdz_local / (1 + z_range_local) 

    N_k = R0 * V_local * scipy.integrate.trapezoid(y_z_max, z_range_max) / scipy.integrate.trapezoid(y_z_local, z_range_local) 
    if time_window is not None:
            #check if its in [yr]
            if not isinstance(time_window, u.Quantity):
                "Time window given has no specified unit. Assuming [yr]"
                time_window = time_window * u.yr
            #change to yr if different
            time_window = time_window.to(u.yr)
            return N_k * time_window
    return N_k

In [ ]:
def rate_integral(phi_z, z_max=10, p=p_cosmology, t_obs=1*u.yr):

    cosmo = acosmo.LambdaCDM(H0=p['h0']* u.km / u.s / u.Mpc, 
                            Om0=p['omega_M'],
                            Ode0=p['omega_L'])#, Tcmb0=2.725 * u.K,


    z_range = np.linspace(p['z_min'], z_max, 10000)
    dVdz = cosmo.differential_comoving_volume(z_range)
    y = phi_z(z_range) * dVdz

    return scipy.integrate.trapezoid(y, z_range)

def get_phi_z1(phi_z, R0, z_max=10, p=p_cosmology, gamma=p_cosmology['nu_diff_gamma'], t_obs=1*u.yr):
    R0 = R0.to(u.Mpc**-3 * u.yr ** -1) 
    cosmo = acosmo.LambdaCDM(H0=p['h0']* u.km / u.s / u.Mpc, 
                            Om0=p['omega_M'],
                            Ode0=p['omega_L'])#, Tcmb0=2.725 * u.K,
    norm = rate_integral(phi_z, z_max=z_max, p=p, t_obs=t_obs)

    #source integral
    z_range = np.linspace(p['z_min'], z_max, 100000)
    dVdz = cosmo.differential_comoving_volume(z_range)
    d_L = cosmo.luminosity_distance(z_range).to(u.cm)
    dNdz = phi_z(z_range)

    #debug
    print('dVdz: ', dVdz)
    print('d_L: ', d_L)
    print('dNdz: ', dNdz)

    y = (1 + z_range) ** (gamma + 2) / (d_L ** 2) * dNdz * dVdz / norm

    print('y: ', y)

    source_int = scipy.integrate.trapezoid(y, z_range)

    print('source_int: ', source_int)
    #combine all
    nu_diff_phi = p['nu_diff_phi'] *u.GeV ** -1 * u.cm ** -2 * u.s ** -1 * u.sr ** -1#GeV/cm2/s/sr.

    
    N_tot_z1 = get_N_k_firesong(phi_z, R0, z_max=1, time_window=t_obs)
    d_L_z1 = cosmo.luminosity_distance(1).to(u.cm)
    print('d_L_z1: ', d_L_z1**2)

    phi_1 = nu_diff_phi * 4 * np.pi * u.sr/ (N_tot_z1 * d_L_z1 ** 2 * source_int) 
    return phi_1

def get_phi(z, phi_z, R0, gamma=p_cosmology['nu_diff_gamma'], z_max=10, p=p_cosmology, t_obs=1*u.yr):
    '''
    Obtains the neutrino flux normalization of a source at a redshift z
    '''
    phi_1 = get_phi_z1(phi_z, R0, z_max=z_max, p=p, t_obs=t_obs)

    cosmo = acosmo.LambdaCDM(H0=p['h0']* u.km / u.s / u.Mpc, 
                            Om0=p['omega_M'],
                            Ode0=p['omega_L'])#, Tcmb0=2.725 * u.K,
    d_L_z1 = cosmo.luminosity_distance(1)
    d_L = cosmo.luminosity_distance(z)


    phi_z = phi_1 * (d_L_z1 / d_L) ** 2 * (1+z) ** (gamma + 2)

    return phi_z

    



In [5]:
def  E_nu_iso(phi_k_1, gamma=2.53, E_0=1e5 * u.GeV, E_min=3, E_max=8, p=p_cosmology):

    #get cosmology stuff
    cosmo = acosmo.LambdaCDM(H0=p['h0']* u.km / u.s / u.Mpc, 
                            Om0=p['omega_M'],
                            Ode0=p['omega_L'])#, Tcmb0=2.725 * u.K,
    d_L = cosmo.luminosity_distance(1).to(u.cm)
    E_range = np.logspace(E_min, E_max, 10000) * u.GeV
    I_e_func = E_range * (E_range / E_0) ** (-1 * gamma)
    I_e = scipy.integrate.trapezoid(I_e_func, E_range)
    print('\n I_e after int: ', I_e)

    return phi_k_1 * 4 * np.pi * d_L ** 2 * I_e

In [6]:
phi_k_1 = get_phi(1, phi_0, R0=p_cosmology['R0']['SN CC'])
print('\nphi_k(z=1): ', phi_k_1)
print('E_nu isotropic: ', E_nu_iso(phi_k_1).to(u.erg / u.yr))

dVdz:  [8.79916328e+02 3.51933268e+03 7.91775010e+03 ... 2.01446297e+10
 2.01444462e+10 2.01442627e+10] Mpc3 / sr
d_L:  [1.37260166e+24 2.74541294e+24 4.11843382e+24 ... 3.26985172e+29
 3.26988881e+29 3.26992590e+29] cm
dNdz:  [1.01765474e-05 1.01792804e-05 1.01820138e-05 ... 3.78237486e-06
 3.78227525e-06 3.78217565e-06]
y:  [4.32765635e-58 4.32752350e-58 4.32739062e-58 ... 1.86496291e-62
 1.86484569e-62 1.86472847e-62] 1 / cm2
source_int:  5.504609133752722e-58 1 / cm2
d_L_z1:  4.40596953688344e+56 cm2

phi_k(z=1):  1.5288791920736925e-24 1 / (GeV s cm2)

 I_e after int:  216147870872.3333 GeV2
E_nu isotropic:  9.25102483527517e+49 erg / yr


Now we can compare this to the Firesong number

In [7]:
time = (40 * u.d).to(u.s).value
local_density = p_cosmology['R0']['SN CC'].to(u.Mpc**-3 * u.yr ** -1).value
sim_values = firesong_simulation('./', density=local_density, Evolution='CC2015SNR', zmax=10.0,
                    fluxnorm=1.44e-18, index=2.52, LF='SC', Transient=True, filename=None, timescale=time)

##############################################################################
##### FIRESONG initializing  - Calculating Neutrino CDFs #####
Standard candle sources+
Transient Sources, time scale = 3456000.0s
Source evolution assumed: CC2015SNR
Local density of neutrino sources: 0.00010099999999999999/Mpc^3
Total number of neutrinos sources in the Universe: 244761013
Redshift range: 0 - 10.0
Standard Candle Luminosity: 1.7722e+39 erg/yr
##### FIRESONG initialization done #####

Actual diffuse flux simulated :
E^2 dNdE = 1.905077862617426e-18 (E/100 TeV)^(-0.52) [GeV/cm^2.s.sr]


In [8]:
time = (100 * u.d).to(u.s).value
local_density = p_cosmology['R0']['SN CC'].to(u.Mpc**-3 * u.yr ** -1).value
sim_values = firesong_simulation('./', density=local_density, Evolution='CC2015SNR', zmax=10.0,
                    fluxnorm=1.44e-8, index=2.52, LF='SC', Transient=True, filename=None, timescale=time)

##############################################################################
##### FIRESONG initializing  - Calculating Neutrino CDFs #####
Standard candle sources+
Transient Sources, time scale = 8640000.0s
Source evolution assumed: CC2015SNR
Local density of neutrino sources: 0.00010099999999999999/Mpc^3
Total number of neutrinos sources in the Universe: 244761013
Redshift range: 0 - 10.0
Standard Candle Luminosity: 7.0886e+48 erg/yr
##### FIRESONG initialization done #####

Actual diffuse flux simulated :
E^2 dNdE = 1.9981929005049455e-08 (E/100 TeV)^(-0.52) [GeV/cm^2.s.sr]


In [9]:
E = 1e50 * u.GeV * u.s ** -1

In [10]:
E.to(u.erg * u.s ** -1)

<Quantity 1.60217663e+47 erg / s>